# 02 — Agent Decision Analysis
LangGraph execution, MCTS COA, Bayesian threat, D* Lite, VRP

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from agents.graph import BattleTwinAgentGraph

graph = BattleTwinAgentGraph()
state = {
    'units': {'B01': {'uid': 'B01', 'callsign': 'WARHORSE-1', 'lat': 34.05, 'lon': -117.45,
                       'ammo_pct': 60, 'fuel_pct': 70, 'water_pct': 80}},
    'contacts': {'R01': {'uid': 'R01', 'lat': 34.3, 'lon': -117.15, 'confidence': 0.8}},
    'objectives': [{'name': 'OBJ ALPHA'}]
}
result = graph.run(state)
print(f'Agents completed: {list(result.get("agent_outputs", {}).keys())}')
print(f'Threat level: {result.get("threat_level", "N/A")}')

In [ ]:
from planning.mcts_coa import MCTSCourseOfAction
import plotly.graph_objects as go

mcts = MCTSCourseOfAction()
sim_state = {'force_ratio': 2.4, 'terrain_score': 0.6, 'logistics_sustainability': 0.8}
coas = mcts.generate_coas(sim_state, n_coas=5, n_simulations=500)

for coa in coas:
    print(f'{coa.coa_id}: {coa.actions[:3]} score={coa.score:.3f}')

In [ ]:
# COA Comparison Spider Chart
fig = go.Figure()
categories = ['Feasibility', 'Acceptability', 'Suitability', 'Risk', 'Time', 'Resources']
for coa in coas[:3]:
    vals = [coa.feasibility, coa.acceptability, coa.suitability,
            1-coa.risk, coa.time_factor, 1-coa.resource_cost]
    fig.add_trace(go.Scatterpolar(r=vals, theta=categories, fill='toself', name=coa.coa_id))
fig.update_layout(polar=dict(bgcolor='#0a0a0a',
    radialaxis=dict(gridcolor='#003300'), angularaxis=dict(gridcolor='#003300')),
    paper_bgcolor='#0d1117', font=dict(color='#00ff41'), title='COA COMPARISON')
fig.show()

In [ ]:
from planning.threat_assessor import BayesianThreatAssessor
import numpy as np, plotly.express as px

assessor = BayesianThreatAssessor()
contacts = [{'lat': 34.3, 'lon': -117.15, 'confidence': 0.85},
            {'lat': 34.28, 'lon': -117.2, 'confidence': 0.6}]
threat_map = assessor.get_threat_map((50, 50), contacts)
fig = px.imshow(threat_map, color_continuous_scale=[[0,'#003300'],[0.5,'#ffcc00'],[1,'#ff3333']],
                title='BAYESIAN THREAT HEATMAP')
fig.show()

In [ ]:
from planning.dstar_lite import DStarLitePlanner
import numpy as np, plotly.graph_objects as go

planner = DStarLitePlanner()
grid = np.ones((50, 50), dtype=np.float32)
grid[15:35, 25] = 999  # obstacle
path = planner.plan((5, 5), (45, 45), grid)
print(f'Path: {len(path)} cells, cost={path.total_cost:.1f}')

fig = go.Figure()
fig.add_trace(go.Heatmap(z=grid, colorscale=[[0,'#003300'],[1,'#ff3333']], showscale=False))
pr, pc = zip(*path.cells)
fig.add_trace(go.Scatter(x=list(pc), y=list(pr), mode='lines',
                         line=dict(color='#00ff41', width=3), name='D* Lite'))
fig.add_trace(go.Scatter(x=[5,45], y=[5,45], mode='lines',
                         line=dict(color='#ff6600', dash='dash'), name='Straight'))
fig.update_layout(paper_bgcolor='#0d1117', plot_bgcolor='#0a0a0a',
                  font=dict(color='#00ff41'), title='D* LITE vs STRAIGHT LINE')
fig.show()

In [ ]:
from planning.vrp_logistics import VRPLogistics

vrp = VRPLogistics(max_vehicles=3)
locs = [(34.05,-117.45),(34.10,-117.38),(34.15,-117.30),(34.20,-117.25),(34.08,-117.40)]
sol = vrp.solve(locs, unit_ids=['B01','B02','B03','B04','B05'])
print(f'VRP Routes: {sol.routes}')
print(f'Total distance: {sol.total_distance_m/1000:.1f}km')
print(f'Units served: {sol.units_served}')

In [ ]:
from agents.commander_agent import CommanderAgent

cdr = CommanderAgent()
sitrep = cdr.generate_sitrep(result)
print(sitrep)
print(f'\nMission accomplishment: {cdr.assess_mission_accomplishment(result):.2f}')